# NB_bishop_ch16_figures

**Bishop Chapter 16 — Key Figures: PCA, Projections, Scree Plot, Probabilistic PCA, Reconstruction**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 16 on continuous latent variables.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_16/NB_bishop_ch16_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 16.1 — PCA on 2D Data: Principal Components as Arrows

In [ ]:
np.random.seed(42)

# Correlated 2D data
N = 150
mean = np.array([2, 3])
cov = np.array([[2.5, 1.8],
                [1.8, 1.5]])
X = np.random.multivariate_normal(mean, cov, N)

# PCA via eigen-decomposition of sample covariance
X_centered = X - X.mean(axis=0)
C = np.cov(X_centered, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(C)

# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X[:, 0], X[:, 1], s=20, alpha=0.4, color='gray', edgecolors='none')

# Draw eigenvectors scaled by sqrt(eigenvalue)
origin = X.mean(axis=0)
colors_pc = [COLORS['red'], COLORS['blue']]
labels_pc = ['PC1', 'PC2']

for i in range(2):
    vec = eigvecs[:, i] * np.sqrt(eigvals[i]) * 2  # scale for visibility
    ax.annotate('', xy=origin + vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=colors_pc[i], lw=3,
                                mutation_scale=20))
    ax.text(origin[0] + vec[0] * 1.15, origin[1] + vec[1] * 1.15,
            f'{labels_pc[i]}\n$\\lambda_{i+1}={eigvals[i]:.2f}$',
            fontsize=11, color=colors_pc[i], ha='center', fontweight='bold')

# Draw covariance ellipse (2-sigma)
angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
width, height = 2 * 2 * np.sqrt(eigvals)
ell = matplotlib.patches.Ellipse(origin, width, height, angle=angle,
                                  fill=False, edgecolor=COLORS['amber'],
                                  lw=2, ls='--', label='$2\\sigma$ ellipse')
ax.add_patch(ell)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('PCA: Principal Components as Eigenvectors of Covariance')
ax.set_aspect('equal')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig16_1_pca_2d')
plt.show()

## Figure 16.3 — Projection onto Principal Subspace

In [ ]:
np.random.seed(42)

# Correlated 2D data
N_proj = 40
mean_proj = np.array([0, 0])
cov_proj = np.array([[3.0, 2.0],
                      [2.0, 2.0]])
X_proj = np.random.multivariate_normal(mean_proj, cov_proj, N_proj)

# PCA
X_c = X_proj - X_proj.mean(axis=0)
C_proj = np.cov(X_c, rowvar=False)
eigvals_p, eigvecs_p = np.linalg.eigh(C_proj)
idx_p = np.argsort(eigvals_p)[::-1]
eigvals_p = eigvals_p[idx_p]
eigvecs_p = eigvecs_p[:, idx_p]

# First principal component direction
u1 = eigvecs_p[:, 0]
origin_p = X_proj.mean(axis=0)

# Project onto PC1
projections = X_c @ u1[:, None] @ u1[None, :] + origin_p

fig, ax = plt.subplots(figsize=(8, 6))

# Draw the principal subspace line
t = np.linspace(-5, 5, 100)
line = origin_p[None, :] + t[:, None] * u1[None, :]
ax.plot(line[:, 0], line[:, 1], color=COLORS['red'], lw=2, ls='-',
        label='Principal subspace (PC1)', zorder=1)

# Original points
ax.scatter(X_proj[:, 0], X_proj[:, 1], s=40, color=COLORS['blue'],
           edgecolors='k', linewidth=0.5, zorder=5, label='Original data')

# Projected points
ax.scatter(projections[:, 0], projections[:, 1], s=40, color=COLORS['green'],
           edgecolors='k', linewidth=0.5, zorder=5, marker='s',
           label='Projected points')

# Projection lines
for i in range(N_proj):
    ax.plot([X_proj[i, 0], projections[i, 0]],
            [X_proj[i, 1], projections[i, 1]],
            color=COLORS['amber'], lw=0.8, alpha=0.6, zorder=2)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Projection onto Principal Subspace')
ax.set_aspect('equal')
ax.set_xlim(-5, 5)
ax.set_ylim(-4, 4)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig16_3_pca_projection')
plt.show()

## Figure 16.5 — Scree Plot: Eigenvalues and Cumulative Variance Explained

In [ ]:
np.random.seed(7)

# Synthetic high-dimensional data with decaying eigenvalues
D = 15
N_scree = 200

# Create data with known eigenvalue structure (exponential decay)
true_eigvals = 5.0 * np.exp(-0.35 * np.arange(D)) + 0.1
# Build covariance with these eigenvalues
Q, _ = np.linalg.qr(np.random.randn(D, D))  # random orthogonal matrix
Sigma_scree = Q @ np.diag(true_eigvals) @ Q.T
X_scree = np.random.multivariate_normal(np.zeros(D), Sigma_scree, N_scree)

# PCA
X_sc = X_scree - X_scree.mean(axis=0)
C_scree = np.cov(X_sc, rowvar=False)
eigvals_s = np.linalg.eigvalsh(C_scree)[::-1]

variance_explained = eigvals_s / eigvals_s.sum()
cumulative_var = np.cumsum(variance_explained)

components = np.arange(1, D + 1)

fig, ax1 = plt.subplots(figsize=(9, 5))

# Bar chart of individual variance explained
bars = ax1.bar(components, variance_explained * 100, color=COLORS['blue'],
               alpha=0.7, edgecolor='k', linewidth=0.5, label='Individual')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)', color=COLORS['blue'])
ax1.tick_params(axis='y', labelcolor=COLORS['blue'])

# Cumulative on secondary axis
ax2 = ax1.twinx()
ax2.plot(components, cumulative_var * 100, 'o-', color=COLORS['red'], lw=2,
         ms=6, markeredgecolor='k', markeredgewidth=0.5, label='Cumulative')
ax2.set_ylabel('Cumulative Variance (%)', color=COLORS['red'])
ax2.tick_params(axis='y', labelcolor=COLORS['red'])
ax2.set_ylim(0, 105)

# Threshold line at 90%
ax2.axhline(90, ls='--', color=COLORS['green'], lw=1.5)
ax2.text(D - 1, 91.5, '90% threshold', fontsize=9, color=COLORS['green'],
         ha='right', style='italic')

# Find elbow
n_90 = np.argmax(cumulative_var >= 0.9) + 1
ax1.axvline(n_90 + 0.5, ls=':', color=COLORS['amber'], lw=1.5)
ax1.text(n_90 + 0.7, variance_explained[0] * 80,
         f'{n_90} components\nexplain 90%',
         fontsize=9, color=COLORS['amber'], style='italic')

ax1.set_title('Scree Plot')
ax1.set_xticks(components)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

fig.tight_layout()
save_fig(fig, 'fig16_5_scree_plot')
plt.show()

## Figure 16.6 — Probabilistic PCA Generative Model Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-1, 11)
ax.set_ylim(-1.5, 4.5)
ax.set_aspect('equal')
ax.axis('off')

box_kw = dict(boxstyle='round,pad=0.5', lw=2)
arrow_kw = dict(arrowstyle='->', lw=2.5, mutation_scale=18)

# ── Latent variable z ───────────────────────────────────────
z_x, z_y = 1.5, 2.0
ax.text(z_x, z_y, '$\\mathbf{z}$', fontsize=20, ha='center', va='center',
        bbox=dict(**box_kw, fc='#e8edf3', ec=COLORS['blue']))
ax.text(z_x, z_y + 1.2, '$\\mathbf{z} \\sim \\mathcal{N}(\\mathbf{0}, \\mathbf{I}_M)$',
        fontsize=12, ha='center', va='center', color=COLORS['blue'])
ax.text(z_x, z_y - 1.0, 'Latent\n($M$-dim)', fontsize=9,
        ha='center', va='center', color='gray', style='italic')

# ── Mapping W ───────────────────────────────────────────────
w_x, w_y = 5.0, 2.0
rect_w = FancyBboxPatch((w_x - 1.2, w_y - 0.6), 2.4, 1.2,
                          boxstyle='round,pad=0.15',
                          fc='#fff3e0', ec=COLORS['amber'], lw=2)
ax.add_patch(rect_w)
ax.text(w_x, w_y, '$\\mathbf{W}\\mathbf{z} + \\boldsymbol{\\mu}$',
        fontsize=14, ha='center', va='center', color=COLORS['amber'],
        fontweight='bold')
ax.text(w_x, w_y + 1.2, 'Linear\nMapping', fontsize=10,
        ha='center', va='center', color=COLORS['amber'])

# ── Noise ε ─────────────────────────────────────────────────
eps_x, eps_y = 5.0, -0.5
ax.text(eps_x, eps_y, '$\\boldsymbol{\\epsilon}$', fontsize=18,
        ha='center', va='center',
        bbox=dict(**box_kw, fc='#fce4ec', ec=COLORS['red']))
ax.text(eps_x + 2.0, eps_y,
        '$\\boldsymbol{\\epsilon} \\sim \\mathcal{N}(\\mathbf{0}, \\sigma^2 \\mathbf{I}_D)$',
        fontsize=12, ha='left', va='center', color=COLORS['red'])

# ── Addition node ───────────────────────────────────────────
add_x, add_y = 7.5, 2.0
circle = plt.Circle((add_x, add_y), 0.4, fc='#e8f5e9',
                      ec=COLORS['green'], lw=2)
ax.add_patch(circle)
ax.text(add_x, add_y, '$+$', fontsize=20, ha='center', va='center',
        color=COLORS['green'], fontweight='bold')

# ── Observed variable x ─────────────────────────────────────
x_x, x_y = 9.5, 2.0
ax.text(x_x, x_y, '$\\mathbf{x}$', fontsize=20, ha='center', va='center',
        bbox=dict(**box_kw, fc='#e8f5e9', ec=COLORS['green']))
ax.text(x_x, x_y - 1.0, 'Observed\n($D$-dim)', fontsize=9,
        ha='center', va='center', color='gray', style='italic')

# ── Arrows ──────────────────────────────────────────────────
# z -> W
ax.annotate('', xy=(w_x - 1.2, w_y), xytext=(z_x + 0.6, z_y),
            arrowprops=dict(**arrow_kw, color=COLORS['blue']))
# W -> +
ax.annotate('', xy=(add_x - 0.4, add_y), xytext=(w_x + 1.2, w_y),
            arrowprops=dict(**arrow_kw, color=COLORS['amber']))
# ε -> +
ax.annotate('', xy=(add_x, add_y - 0.4), xytext=(eps_x + 0.5, eps_y + 0.3),
            arrowprops=dict(**arrow_kw, color=COLORS['red']))
# + -> x
ax.annotate('', xy=(x_x - 0.55, x_y), xytext=(add_x + 0.4, add_y),
            arrowprops=dict(**arrow_kw, color=COLORS['green']))

# Equation at bottom
ax.text(5.0, -1.2,
        '$\\mathbf{x} = \\mathbf{W}\\mathbf{z} + \\boldsymbol{\\mu} + \\boldsymbol{\\epsilon}, '
        '\\quad p(\\mathbf{x}|\\mathbf{z}) = '
        '\\mathcal{N}(\\mathbf{W}\\mathbf{z} + \\boldsymbol{\\mu},\\, \\sigma^2\\mathbf{I})$',
        fontsize=13, ha='center', va='center')

ax.set_title('Probabilistic PCA — Generative Model', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig16_6_ppca')
plt.show()

## Figure 16.8 — PCA Reconstruction with Increasing Number of Components

In [ ]:
np.random.seed(42)

# Synthetic high-dimensional signal: sum of sinusoids + noise
D_recon = 50
t = np.linspace(0, 2 * np.pi, D_recon)

# True signal: superposition of a few frequencies
N_recon = 200
freqs = [1, 3, 5]
amplitudes = [3.0, 1.5, 0.8]

X_recon = np.zeros((N_recon, D_recon))
for i in range(N_recon):
    phases = np.random.uniform(0, 2 * np.pi, len(freqs))
    signal = sum(a * np.sin(f * t + p) for a, f, p in zip(amplitudes, freqs, phases))
    X_recon[i] = signal + 0.5 * np.random.randn(D_recon)

# PCA
X_mean = X_recon.mean(axis=0)
X_cent = X_recon - X_mean
U, S, Vt = np.linalg.svd(X_cent, full_matrices=False)

# Reconstruct a single example with different numbers of components
example_idx = 0
x_orig = X_recon[example_idx]
x_cent = X_cent[example_idx]

n_components_list = [1, 2, 5, 10]
cols_recon = [COLORS['red'], COLORS['amber'], COLORS['green'], COLORS['blue']]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, n_comp, col in zip(axes.flat, n_components_list, cols_recon):
    # Reconstruct
    z = x_cent @ Vt[:n_comp].T  # project
    x_hat = z @ Vt[:n_comp] + X_mean  # reconstruct

    # Reconstruction error
    mse = np.mean((x_orig - x_hat) ** 2)

    ax.plot(t, x_orig, color='gray', lw=1.5, alpha=0.5, label='Original')
    ax.plot(t, x_hat, color=col, lw=2, label=f'{n_comp} component{"s" if n_comp > 1 else ""}')

    ax.set_xlabel('$t$')
    ax.set_ylabel('$x(t)$')
    ax.set_title(f'$M = {n_comp}$ (MSE = {mse:.3f})')
    ax.legend()

fig.suptitle('PCA Reconstruction with Increasing Number of Components', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, 'fig16_8_reconstruction')
plt.show()

In [ ]:
print('All Chapter 16 figures generated.')